# SCD (Slowly Changing Dimension)

- In data engineering, "Dimensions" are things like Customers, Products, or Locations. 
- These things change slowly over time (e.g., a customer changes their phone number or address).
- **SCD is simply the strategy we use to handle those changes in our database.**

# SCD Type-0:
- No update of exisitng records.
- Only existing records will be inserted.

Example: Mostly sceanrio where one time data load required or Static Table are comes under SCD-Tpye: 0


# SCD Type-1:
- here Insertion and Update will take place, if it is a new record then will be inserted directly else update the exisitng record.
- History will not be maintained
- changes in dimension data by **directly overwriting existing records** with new information, storing no history

## SCD Type-1 in Delta Table using python

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# 1. Define the schema
schema = StructType([
    StructField("Customer_ID", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("City", StringType(), True)
])

# 2. Create the Target Table (Initial State)
# Rahul lives in Chennai.
data_initial = [(101, "Rahul", "Chennai"), (102, "Anjali", "Mumbai")]
df_initial = spark.createDataFrame(data_initial, schema)


In [0]:
# 3. Save as a Delta Table
spark.sql("CREATE DATABASE IF NOT EXISTS workspace.default")
df_initial.write.format("delta").mode("overwrite").saveAsTable("workspace.default.dim_customers")

print("--- Initial Table ---")
spark.table("workspace.default.dim_customers").show()


--- Initial Table ---
+-----------+------+-------+
|Customer_ID|  Name|   City|
+-----------+------+-------+
|        101| Rahul|Chennai|
|        102|Anjali| Mumbai|
+-----------+------+-------+



The Updates (The Incoming Change)

In [0]:
# 4. Create the Updates DataFrame (Source)
# Rahul moves to Bangalore (Update).
# Vikram is a new customer (Insert).
data_updates = [(101, "Rahul", "Bangalore"), (103, "Vikram", "Delhi")]
updates_df = spark.createDataFrame(data_updates, schema)

print("--- Incoming Updates ---")
updates_df.show()


--- Incoming Updates ---
+-----------+------+---------+
|Customer_ID|  Name|     City|
+-----------+------+---------+
|        101| Rahul|Bangalore|
|        103|Vikram|    Delhi|
+-----------+------+---------+



**Upsert=>Update + Insert in Python Syntax**

deltaTable_Obj.merge(source=new_Table,condition="")
    .whenMatchedUpdate(set={})
    .whenNotMatchedInsert(values={})
    .execute()

Pass as dictionay what need to be inserted or Updated.   

In [0]:
from delta.tables import *

# 5. Initialize the Delta Table object
# Step-1: reading the existing Delta table
target_table = DeltaTable.forName(spark, "workspace.default.dim_customers")

# 6. Execute the Merge (Upsert)
target_table.alias("target") \
  .merge(
    source = updates_df.alias("source"),
    condition = "target.Customer_ID = source.Customer_ID"
  ) \
  .whenMatchedUpdate(set = {
    "Name": "source.Name",
    "City": "source.City"
  }) \
  .whenNotMatchedInsert(values = {
    "Customer_ID": "source.Customer_ID",
    "Name": "source.Name",
    "City": "source.City"
  }) \
  .execute()



DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
print("--- Final Table (After SCD Type 1) ---")
target_table.toDF().show()


--- Final Table (After SCD Type 1) ---
+-----------+------+---------+
|Customer_ID|  Name|     City|
+-----------+------+---------+
|        102|Anjali|   Mumbai|
|        101| Rahul|Bangalore|
|        103|Vikram|    Delhi|
+-----------+------+---------+



## SCD Type-1 in Delta Table using SQL
**Upsert=>Update + Insert in SQL Syntax**

In [0]:
# Register as a temporary view for SQL usage
updates_df.createOrReplaceTempView("source_updates")

In [0]:
%sql
MERGE INTO workspace.default.dim_customers AS target
USING source_updates AS source
ON target.Customer_ID = source.Customer_ID
WHEN MATCHED THEN
  UPDATE SET 
    target.Name = source.Name,
    target.City = source.City
WHEN NOT MATCHED THEN
  INSERT (Customer_ID, Name, City)
  VALUES (source.Customer_ID, source.Name, source.City)


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,2,0,0


In [0]:
spark.sql("select * from workspace.default.dim_customers").show()

+-----------+------+---------+
|Customer_ID|  Name|     City|
+-----------+------+---------+
|        102|Anjali|   Mumbai|
|        101| Rahul|Bangalore|
|        103|Vikram|    Delhi|
+-----------+------+---------+



## Simple Example for SCD-1

In [0]:
import pyspark
from pyspark.sql import SparkSession as ss

In [0]:
spark=ss.builder.appName('SCD-1').getOrCreate()

In [0]:
data=[
    [101,"Vijay","Mumbai"],
    [102,'Aijth','Chennai'],
    [103,'Karthi','Kerala']
]
columns=['Id','Name','City']

df=spark.createDataFrame(data,columns)
df.show()

+---+------+-------+
| Id|  Name|   City|
+---+------+-------+
|101| Vijay| Mumbai|
|102| Aijth|Chennai|
|103|Karthi| Kerala|
+---+------+-------+



In [0]:
daily_data=[
    [101,'Vijay','Dubai'],
    [104,'Vikram','CBE'],
    [105,'Rajini','TN']
]

daily_df=spark.createDataFrame(daily_data,columns)
daily_df.show()

+---+------+-----+
| Id|  Name| City|
+---+------+-----+
|101| Vijay|Dubai|
|104|Vikram|  CBE|
|105|Rajini|   TN|
+---+------+-----+



In [0]:
df_outer_jn=df.join(daily_df,'id','full_outer')
display(df_outer_jn)

Id,Name,City,Name,City
101,Vijay,Mumbai,Vijay,Dubai
102,Aijth,Chennai,null,null
103,Karthi,Kerala,null,null
105,null,null,Rajini,TN
104,null,null,Vikram,CBE


here Coalesce ==>  is used to merge two columns by picking the first non-null value found between them

In [0]:
from pyspark.sql.functions import coalesce

In [0]:
res_df=df.join(daily_df,'id','full_outer').select(
    coalesce(df.Id,daily_df.Id).alias("Id"),
    coalesce(df.Name,daily_df.Name).alias("Name"),
    coalesce(df.City,daily_df.City).alias("City")
)
# display(df)

In [0]:
display(res_df)

Id,Name,City
101,Vijay,Mumbai
102,Aijth,Chennai
103,Karthi,Kerala
105,Rajini,TN
104,Vikram,CBE
